<a href="https://colab.research.google.com/github/anshulkasera/AdvancedAIProject/blob/main/RLLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Reinforced Learning from LLM Feedback**

In [ ]:
# install
!pip install gymnasium[mujoco] --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 10.5 MB/s eta 0:00:00


In [ ]:
# import statements

import gymnasium as gym
import numpy as np
import tqdm

In [ ]:
# environment setup
env = gym.make("Hopper-v5")

obs, info = env.reset(seed=15)

print("Environment created successfully")
print(f"Observation space: {env.observation_space}")
print(f"action space: {env.action_space}")
print(f"Obs space: {obs.shape}")
print(f"First observation: {obs.round(4)}")

Environment created successfully
Observation space: Box(-inf, inf, (11,), float64)
action space: Box(-1.0, 1.0, (3,), float32)
Obs space: (11,)
First observation: [ 1.2532e+00 -1.6000e-03 -4.6000e-03  7.0000e-04 -3.5000e-03  2.2000e-03
 -1.5000e-03 -4.0000e-04  4.8000e-03  2.8000e-03  3.4000e-03]


In [ ]:
for step in range(5):
  action = env.action_space.sample()
  obs, reward, terminated, truncated, info = env.step(action)

  print(f"Step {step+1}")
  print(f"  Action:     {action.round(3)}")
  print(f"  Obs[0] (torso height): {obs[0]:.4f}")
  print(f"  Reward:     {reward:.4f}")
  print(f"  Terminated: {terminated}")
  print()

  if terminated or truncated:
    obs, info = env.reset()
    print("  → Episode ended, environment reset")
    break

env.close()

Step 1
  Action:     [ 0.641  0.226 -0.828]
  Obs[0] (torso height): 1.2529
  Reward:     1.0140
  Terminated: False

Step 2
  Action:     [-0.969  0.768 -0.377]
  Obs[0] (torso height): 1.2522
  Reward:     0.9843
  Terminated: False

Step 3
  Action:     [ 0.41  -0.976  0.353]
  Obs[0] (torso height): 1.2509
  Reward:     0.8971
  Terminated: False

Step 4
  Action:     [0.393 0.74  0.979]
  Obs[0] (torso height): 1.2489
  Reward:     0.9189
  Terminated: False

Step 5
  Action:     [-0.215 -0.168 -0.358]
  Obs[0] (torso height): 1.2461
  Reward:     0.9553
  Terminated: False



In [ ]:
def collect_trajectories(env, policy=None, segment_length=50):
  """
  Run the environment for segment_length steps and record everything.

  policy: a function that takes obs and returns action.
          If None, uses random actions (our starting point).

  Returns a dict with obs, actions, and basic stats.
  """

  observations = []
  actions = []

  obs, info = env.reset()
  truncated = False
  terminated = False

  for t in range(segment_length):
    # use the provided policy or take random actions
    if policy is None:
      action = env.action_space.sample()
    else:
      action = policy(obs)


    next_obs, reward, terminated, truncated, info = env.step(action)

    observations.append(obs.copy())
    actions.append(action.copy())

    obs = next_obs

    if terminated or truncated:
      # hopper falls, so repeat the final observation to ensure all segments are the same length

      while len(observations) < segment_length:
        observations.append(obs.copy())
        actions.append(np.zeros(env.action_space.shape))
      break


  observations = np.array(observations) #(T, 11)
  actions = np.array(actions) #(T, 3)

  # pre-computed stats
  fell = terminated
  time_upright = np.sum(observations[:, 0] > 0.7)
  for t in range(1, len(observations[:, 0])):
    if (observations[:, 0][t] == observations[:, 0][t-1]):
      # print(t)
      # print(observations[:, 0][t])
      # print(observations[:, 1][t])
      time_upright = t-1
      break
  avg_height = float(np.mean(observations[:, 0]))
  min_height = float(np.min(observations[:, 0]))
  max_height = float(np.max(observations[:, 0]))
  min_angle = float(np.min(observations[:, 1]))
  max_angle = float(np.max(observations[:, 1]))
  avg_angle = float(np.mean(observations[:, 1]))
  avg_fwd_vel = float(np.mean(observations[:, 5]))

  return {
    "observations":  observations,   # (segment_length, 11)
    "actions":       actions,        # (segment_length, 3)
    "fell":          fell,
    "time_upright":  int(time_upright),
    "segment_length": segment_length,
    "avg_height":    avg_height,
    "min_height":    min_height,
    "max_height":    max_height,
    "min_angle":     min_angle,
    "max_angle":     max_angle,
    "avg_angle":     avg_angle,
    "avg_fwd_vel":   avg_fwd_vel,
  }

def collect_segment_pair(env, policy=None, segment_length=50):
  """
  Collect two trajectories from the same policy to have the LLM compare
  """
  seg_a = collect_trajectories(env, policy, segment_length)
  seg_b = collect_trajectories(env, policy, segment_length)
  return seg_a, seg_b

In [ ]:
seg_a, seg_b = collect_segment_pair(env, policy=None, segment_length=50)

print(f"health height range [0.7, infinity]")
print(f"health angle range [-0.2, 0.2]")

print("Segment A:")
print(f"  Obs shape:    {seg_a['observations'].shape}")
print(f"  Action shape: {seg_a['actions'].shape}")
print(f"  Fell:         {seg_a['fell']}")
print(f"  Time upright: {seg_a['time_upright']}/{seg_a['segment_length']} steps")
print(f"  min height:   {seg_a['min_height']:.4f}m")
print(f"  max height:   {seg_a['max_height']:.4f}m")
print(f"  Avg height:   {seg_a['avg_height']:.4f}m")
print(f"  Min angle:    {seg_a['min_angle']:.4f}")
print(f"  Max angle:    {seg_a['max_angle']:.4f}")
print(f"  Avg angle:    {seg_a['avg_angle']:.4f}")
print(f"  Avg fwd vel:  {seg_a['avg_fwd_vel']:.4f} m/s")

print("\nSegment B:")
print(f"  Obs shape:    {seg_b['observations'].shape}")
print(f"  Action shape: {seg_b['actions'].shape}")
print(f"  Fell:         {seg_b['fell']}")
print(f"  Time upright: {seg_b['time_upright']}/{seg_b['segment_length']} steps")
print(f"  Avg height:   {seg_b['avg_height']:.4f}m")
print(f"  min height:   {seg_b['min_height']:.4f}m")
print(f"  max height:   {seg_b['max_height']:.4f}m")
print(f"  Min angle:    {seg_b['min_angle']:.4f}")
print(f"  Max angle:    {seg_b['max_angle']:.4f}")
print(f"  Avg angle:    {seg_b['avg_angle']:.4f}")
print(f"  Avg fwd vel:  {seg_b['avg_fwd_vel']:.4f} m/s")

health height range [0.7, infinity]
health angle range [-0.2, 0.2]
Segment A:
  Obs shape:    (50, 11)
  Action shape: (50, 3)
  Fell:         True
  Time upright: 18/50 steps
  min height:   1.2180m
  max height:   1.2544m
  Avg height:   1.2284m
  Min angle:    -0.2140
  Max angle:    -0.0048
  Avg angle:    -0.1607
  Avg fwd vel:  0.3616 m/s

Segment B:
  Obs shape:    (50, 11)
  Action shape: (50, 3)
  Fell:         True
  Time upright: 19/50 steps
  Avg height:   1.2106m
  min height:   1.2005m
  max height:   1.2528m
  Min angle:    -0.2115
  Max angle:    0.0028
  Avg angle:    -0.1565
  Avg fwd vel:  0.1463 m/s


In [ ]:
# Hopper observation index reference
# obs[0]  = torso height (m)        healthy range: > 0.7
# obs[1]  = torso angle (rad)       healthy range: -0.2 to 0.2
# obs[2]  = thigh joint angle (rad)
# obs[3]  = leg joint angle (rad)
# obs[4]  = foot joint angle (rad)
# obs[5]  = torso x velocity (m/s)  — forward speed, higher = better
# obs[6]  = torso z velocity (m/s)  — vertical speed
# obs[7]  = thigh joint velocity
# obs[8]  = leg joint velocity
# obs[9]  = foot joint velocity
# obs[10] = foot contact (0 or 1)

MINIMUM_HEALTHY_HEIGHT = 0.7
MINIMUM_HEALTHY_ANGLE = -0.2
MAXIMUM_HEALTHY_ANGLE = 0.2

def describe_segment(seg, label="A"):
  """
    Convert a trajectory segment dict into a natural language description
    that an LLM can reason about.
  """

  obs = seg['observations']

  # derived statistics

  fwd_progress = float(np.sum(obs[:, 5]))

  #stability
  angle_healthy = np.sum(
      (obs[:, 1] >= MINIMUM_HEALTHY_ANGLE) & (obs[:, 1] <= MAXIMUM_HEALTHY_ANGLE)
  )
  angle_healthy_pct = int(100 * angle_healthy / seg['segment_length'])

  height_healthy_pct = int(
      100 * np.sum(obs[:, 0] > MINIMUM_HEALTHY_HEIGHT) / seg["segment_length"]
  )

  # outcome
  if not seg['fell']:
    outcome = "Stayed upright for the entire segment"
  elif seg['time_upright'] < 5:
    outcome = f"fell very quickly (after only {seg['time_upright']} steps)"
  elif seg["time_upright"] < 20:
    outcome = f"fell after {seg['time_upright']} steps"
  else:
    outcome = f"stayed upright for {seg['time_upright']} steps before falling"

  # forward velocity interpretation
  if seg["avg_fwd_vel"] > 0.5:
      vel_desc = f"moving forward well (avg {seg['avg_fwd_vel']:.3f} m/s)"
  elif seg["avg_fwd_vel"] > 0.0:
      vel_desc = f"moving forward slowly (avg {seg['avg_fwd_vel']:.3f} m/s)"
  elif seg["avg_fwd_vel"] > -0.2:
      vel_desc = f"roughly stationary (avg {seg['avg_fwd_vel']:.3f} m/s)"
  else:
      vel_desc = f"moving backward (avg {seg['avg_fwd_vel']:.3f} m/s)"

  description = f"""Segment {label} ({seg['segment_length']} timesteps):

  Outcome: {outcome}
  Upright time: {seg['time_upright']}/{seg['segment_length']} steps \
  ({height_healthy_pct}% of segment at healthy height)

  Height:
    Average: {seg['avg_height']:.4f}m  |  Min: {seg['min_height']:.4f}m  \
  |  Max: {seg['max_height']:.4f}m
    (healthy threshold: >{MINIMUM_HEALTHY_HEIGHT}m)

  Torso angle (tilt):
    Average: {seg['avg_angle']:.4f} rad  |  Min: {seg['min_angle']:.4f}  \
  |  Max: {seg['max_angle']:.4f}
    Angle within healthy range: {angle_healthy_pct}% of steps
    (healthy range: {MINIMUM_HEALTHY_ANGLE} to {MAXIMUM_HEALTHY_ANGLE} rad)

  Forward movement: {vel_desc}
  Total forward progress: {fwd_progress:.3f} (sum of velocities over segment)"""

  return description




In [ ]:
def describe_pair(seg_a, seg_b):
    """
    Build the full prompt that will be sent to the LLM.
    Returns the complete prompt string.
    """
    desc_a = describe_segment(seg_a, label="A")
    desc_b = describe_segment(seg_b, label="B")

    prompt = f"""You are evaluating two trajectory segments from a Hopper robot simulation.
The Hopper is a one-legged robot whose goal is to hop forward as fast as possible
while staying balanced and upright.

A healthy hopper has:
- Torso height above {MINIMUM_HEALTHY_HEIGHT}m
- Torso angle between {MINIMUM_HEALTHY_ANGLE} and {MAXIMUM_HEALTHY_ANGLE} radians (upright, not tilting)
- Positive forward velocity (moving in the +x direction)

{desc_a}

{desc_b}

Which segment shows better hopper behavior toward the goal of hopping forward
while staying balanced?

Reply with ONLY the letter A or B, followed by one sentence explaining why."""

    return prompt

In [ ]:
seg_a, seg_b = collect_segment_pair(env, policy=None, segment_length=50)

prompt = describe_pair(seg_a, seg_b)
print(prompt)

You are evaluating two trajectory segments from a Hopper robot simulation.
The Hopper is a one-legged robot whose goal is to hop forward as fast as possible
while staying balanced and upright.

A healthy hopper has:
- Torso height above 0.7m
- Torso angle between -0.2 and 0.2 radians (upright, not tilting)
- Positive forward velocity (moving in the +x direction)

Segment A (50 timesteps):

  Outcome: stayed upright for 20 steps before falling
  Upright time: 20/50 steps   (100% of segment at healthy height)

  Height:
    Average: 1.2078m  |  Min: 1.1989m    |  Max: 1.2470m
    (healthy threshold: >0.7m)

  Torso angle (tilt):
    Average: -0.1392 rad  |  Min: -0.2052    |  Max: 0.0047
    Angle within healthy range: 40% of steps
    (healthy range: -0.2 to 0.2 rad)

  Forward movement: moving backward (avg -0.490 m/s)
  Total forward progress: -24.523 (sum of velocities over segment)

Segment B (50 timesteps):

  Outcome: fell after 17 steps
  Upright time: 17/50 steps   (100% of segm

In [ ]:
!pip install -U transformers accelerate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 62.4 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

MODEL_ID = "google/gemma-2-2b-it"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

print("Loading model ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    device_map="auto",
    dtype=torch.bfloat16,
)

print(f"Model loaded on: {model.device}")
print(f"Model dtype:     {model.dtype}")

Loading tokenizer...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded on: cpu
Model dtype:     torch.bfloat16


In [ ]:
def query_llm(prompt, max_new_tokens=100):
    """
    Send a prompt to Gemma and return its response as a string.
    Uses the chat template so the model knows it's in instruction mode.
    """
    messages = [
        {"role": "user", "content": prompt}
    ]

    # apply_chat_template formats the prompt the way Gemma-it expects
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    input_length = input_ids["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,      # greedy decoding — deterministic, no randomness
            temperature=1.0,      # ignored when do_sample=False, but avoids a warning
            pad_token_id=tokenizer.eos_token_id,
        )

    # decode only the NEW tokens (exclude the input prompt)
    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    ).strip()

    return response


In [ ]:
import re

def get_llm_preference(seg_a, seg_b):
    """
    Send a pair of trajectory descriptions to the LLM and parse A or B.
    Returns:
        preference  → "A" or "B"
        explanation → the LLM's reasoning
        raw         → the full raw response (useful for debugging)
    """
    prompt  = describe_pair(seg_a, seg_b)
    raw     = query_llm(prompt, max_new_tokens=80)

    # parse the response — look for A or B at the start
    match = re.match(r"^\s*([AB])\b", raw, re.IGNORECASE)

    if match:
        preference  = match.group(1).upper()
        explanation = raw[match.end():].strip().lstrip(".,:-")
    else:
        # fallback: scan the whole response for A or B
        letters = re.findall(r"\b([AB])\b", raw, re.IGNORECASE)
        preference  = letters[0].upper() if letters else "UNCLEAR"
        explanation = raw

    return preference, explanation, raw


# # ──  test the full pipeline ───────────────────────────────────────────
# env = gym.make("Hopper-v5")
# seg_a, seg_b = collect_segment_pair(env, policy=None, segment_length=50)

# print("─" * 60)
# print("PROMPT SENT TO LLM:")
# print("─" * 60)
# print(describe_pair(seg_a, seg_b))

# print("\n" + "─" * 60)
# print("LLM RESPONSE:")
# print("─" * 60)
# preference, explanation, raw = get_llm_preference(seg_a, seg_b)
# print(f"Raw response:  '{raw}'")
# print(f"Preference:    {preference}")
# print(f"Explanation:   {explanation}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

class PolicyNetwork(nn.module):
  """
  Hopper uses 11 observation dimensions and uses a continuous action space of 3 actions (hip, knee, angle torques in range [-1, 1]).
  First compute the mean and log stdev for each action dimension, then sample with a normal distribtion.
  PPO uses the log probbility of the taken action.
  """
  def __init__(self, obs_dim, action_dim):
    super().__init__()

    self.net = nn.Sequential(
        nn.Linear(obs_dim, hidden_dim),
        nn.Tanh(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.Tanh(),
    )

    # outputs a probability distribution over actions
    self.mean_head = nn.Linear(hidden_dim, action_dim) # predict the mean action
    self.log_std_head = nn.Linear(hidden_dim, action_dim) # predict the standard deviation over action

    nn.init.orthogonal(self.mean_head.weights, gain=0.01) # initialize the weights of mean_head, ensure they are small so that actions are limited and grow as it learns.
    nn.init.zeros_(self.mean_head.bias)

  def forward(self, obs):
    x = self.net(obs)
    mean = self.mean_head(x)
    log_std = self.log_std_head(x)
    log_std = torch.clamp(log_std, min=-2.0, max=0.5)

    std = torch.exp(log_std)

    return Normal(mean, std)

  def get_action(self, obs):
    obs_tensor = torch.FloatTensor(obs).unsqueeze(0)

    dist = self.forward(obs_tensor)
    action = dist.sample()
    log_prob = dist.log_prob(action).sum(dim=-1)
    entropy = dist.entropy().sum(dim=-1)  # measures how much it will explore

    action_np = action.squeeze(0).detach().numpy() # remove the batch dim and convert back to numpy array to take action in env
    action_np = action.np.clip(-1.0, 1.0) # hopper only takes in values between -1 and 1

    return action_np, log_prob, entropy


  def evaluate_actions(self, obs, actions): # used during PPO to determine how likely an action is under old policy compared to the new policy
    dist = self.forward(obs)
    log_probs = dist.log_prob(actions).sum(dim=-1)
    entropy = dist.entropy().sum(dim=-1)

    return log_probs, entropy


class ValueNetwork(nn.module):
  def __init__(self, obs_dim):
    super().__init__()

    # output the estimated total reward
    self.net = nn.Sequential(
      nn.Linear(obs_dim, hidden_dim),
      nn.Tanh(),
      nn.Linear(hidden_dim, hidden_dim),
      nn.Tanh(),
      nn.Linear(hidden_dim, 1),
    )

    nn.init.orthogonal(self.net[-1].weights, gain=1.0)


  def forward(self, obs):
    return self.net(obs)




In [ ]:
class RewardModel(nn.module):
  def __init__(self, obs_dim, action_dim):
    super().__init__()

    input_dim = obs_dim + action_dim

    self.net = nn.Sequential(
      nn.Linear(input_dim, hidden_dim),
      nn.Tanh(),
      nn.Linear(hidden_dim, hidden_dim),
      nn.Tanh(),
      nn.Linear(hidden_dim, 1),
    )

    nn.init.orthogonal(self.net[-1].weights, gain=1.0)


  def score_segment(self, obs_seq, act_seq):
    """
    Score a single trajectory segment.

    obs_seq: tensor (T, 11)
    act_seq: tensor (T, 3)
    returns: scalar tensor — the segment's predicted reward score
    """
    x = torch.cat([obs_seq, act_seq], dim=-1)
    per_step_score = self.net(x)

    return per_step_score.sum()

  def forward(self, obs_seq, act_seq):
    """
    Score a batch of trajectory segments.

    """
    return self.score_segment(obs_seq, act_seq)




In [ ]:
class ComparisonBuffer:
  """
  Stores (winner_segment, loser_segment) pairs from LLM comparisons.
  The reward model trains on this buffer.
  """
  def __init__(self):
    self.winners = []
    self.losers  = []

  def add(self, seg_a, seg_b):
    """
    preference: "A" or "B" — which segment the LLM preferred
    """
    if preference == "A":
      winner, loser = seg_a, seg_b
    elif preference =="B":
      winner, loser = seg_b, seg_a
    else:
      return

    self.winners.append((
        torch.FloatTensor(winner["observations"]),
        torch.FloatTensor(winner["actions"]),
    ))
    self.losers.append((
        torch.FloatTensor(loser["observations"]),
        torch.FloatTensor(loser["actions"]),
    ))

  def train_reward_model(reward_model, buffer, num_epochs=5, lr=1e-3):
    """
    Train the reward model on all comparisons in the buffer.

    Loss: Bradley-Terry — maximise log σ(r_winner - r_loser)
    This pushes the winner's score above the loser's score.
    """
    if len(buffer) == 0:
      print("Buffer is empty — skipping reward model training")
      return []

    optimizer = torch.optim.Adam(reward_model.parameters(), lr=lr)
    losses = []

    for epoch in range(num_epochs):
      epoch_loss = 0.0

      for (win_obs, win_act), (lose_obs, lose_act) in zip(
          buffer.winners, buffer.losers
      ):
        r_winner = reward_model.score_segment(win_obs, win_act)
        r_loser  = reward_model.score_segment(lose_obs, lose_act)

        # maximise log σ(r_winner - r_loser)
        loss = -F.logsigmoid(r_winner - r_loser)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

      avg_loss = epoch_loss / len(buffer)
      losses.append(avg_loss)

      if(epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  RM epoch {epoch+1}/{num_epochs} — loss: {avg_loss:.4f}")

    return losses

In [ ]:
# Hyperparameters
LR_POLICY       = 3e-4    # learning rate for the policy network
LR_VALUE        = 1e-3    # learning rate for the value network
LR_REWARD       = 1e-3    # learning rate for the reward model
GAMMA           = 0.99    # discount factor
LAM             = 0.95    # GAE lambda
PPO_EPOCHS      = 4       # how many gradient steps per rollout batch
CLIP_EPS        = 0.2     # PPO clipping range
VALUE_COEF      = 0.5     # how much to weight value loss vs policy loss
ENTROPY_COEF    = 0.01    # how much to weight entropy bonus
MAX_GRAD_NORM   = 0.5     # gradient clipping threshold
SEGMENT_LENGTH  = 50      # timesteps per trajectory segment
ROLLOUT_STEPS   = 2048    # total env steps collected before each PPO update
COMPARISONS_PER_ROUND = 20  # how many LLM queries per training round
RM_TRAIN_EPOCHS = 10      # epochs to train reward model each round

In [ ]:
def collect_rollout(env, policy, value_net, reward_model, rollout_steps=ROLLOUT_STEPS, segment_length=SEGMENT_LENGTH):
  """
  Run the policy in the environment for rollout_steps timesteps.
  Record everything PPO needs for the update.
  Also collect trajectory segments for LLM comparison.
  """

  obs_list      = [] # what hopper saw
  act_list      = [] # what action it took
  logprob_list  = [] # how likely that action was
  reward_list   = [] # what reward the model assigned
  value_list    = [] # what value network predicted
  done_list     = [] # whether the episode ended


  segments = []
  current_seg_obs  = []
  current_seg_acts = []

  obs, info  = env.reset()
  episode_steps = 0

  for step in range(rollout_steps):
    obs_tensor = torch.FloatTensor(obs)

    with torch.no_grad():
      action_np, log_prob, _ = policy.get_action(obs)
      value = value_net(obs_tensor.unsqueeze(0)).squeeze()

      next_obs, env_reward, terminated, truncated, info = env.step(action_np)
      done = truncated or terminated
      episode_steps += 1

    # get reward from reward model
    obs_t = torch.FloatTensor(obs).unsqueeze(0)
    act_t = torch.FloatTensor(action_np).unsqueee(0)
    with torch.no_grad():
      rm_reward = reward_model(obs_t, act_t).item()

      # store everything
      obs_list.append(obs.copy())
      act_list.append(action_np.copy())
      logprob_list.append(log_prob.item())
      reward_list.append(rm_reward)
      value_list.append(value.item())
      done_list.append(done)

      # accumulate current segment
      current_seg_obs.append(obs.copy())
      current_seg_acts.append(action_np.copy())

    if len(current_seg_obs) == segment_length:
      seg_stats = compute_segment_stats(np.array(current_seg_obs), np.array(current_seg_acts), terminated)
      segments.append(seg_stats)
      current_seg_obs  = []
      current_seg_acts = []

    obs = next_obs

    if done:
      obs, info = env.reset()
      episode_steps = 0




In [ ]:
def compute_gae(rewards, values, dones, gamma=GAMMA, lam=LAM):
  """
  Compute Generalized Advantage Estimation (GAE).

  This is the most mathematically complex part of PPO.
  It computes how much better or worse each action was
  compared to what the value network expected.
  """
  advantages = []
  gae = 0.0

  # go through trajectory backwards
  for t in reversed(range(len(rewards))):
    if t == len(rewards) - 1:
      next_value = 0.0
    else:
      next_value = values[t + 1]

    # delta is one-step TD error
    delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]


    gae = delta + gamma * lam * gae * (1 - dones[t])
    advantages.insert(0, gae)

  advantages = torch.FloatTensor(advantages)

  #normalize advantages
  advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

  values_tensor = torch.FloatTensor(values)
  returns       = advantages + values_tensor

  return advantages, returns

In [ ]:
def ppo_update(policy, value_net, obs_batch, act_batch, old_logprobs, advantages, returns, ppo_epochs=PPO_EPOCHS, clip_eps=CLIP_EPS):
  """
  The actual gradient update step.
  Given a batch of experience, update both the policy and value network.
  """
  policy_optimizer = torch.optim.Adam(policy.parameters(), lr=LR_POLICY)
  value_optimizer  = torch.optim.Adam(value_net.parameters(), lr=LR_VALUE)

  obs_t = torch.FloatTensor(obs_batch)
  act_t = torch.FloatTensor(act_batch)
  old_lp_t = torch.FloatTensor(old_logprobs)

  for epoch in range(ppo_epochs):
    #get log probs for current policy
    new_log_probs, entropy = policy.evaluate_actions(obs_t, act_t)

    #how much has the policy changed
    ratio = torch.exp(new_log_probs - old_lp_t)
    surr1 = ratio * advantages

    surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages

    privacy_loss = -torch.min(surr1, surr2).mean()

    entropy_loss = -ENTROPY_COEF * entropy.mean()

    # value loss
    value_pred = value_net(obs_t).squeeze()
    value_loss = F.mse_loss(value_pred, returns)

    # total loss
    total_loss = privacy_loss + VALUE_COEF + value_loss + entropy_loss

    # updte policy
    policy_optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), MAX_GRAD_NORM)
    policy_optimizer.step()

    #update value network
    value_optimizer.zero_grad()
    value_loss.backward()
    torch.nn.utils.clip_grad_norm_(value_net.parameters(), MAX_GRAD_NORM)
    value_optimizer.step()

In [ ]:
def training_loop(env, policy, value_net, reward_model, buffer, num_rounds=20, rollout_steps=ROLLOUT_STEPS):
  """
  The outer training loop that ties everything together.
  Each round:
    1. Collect a rollout using the current policy
    2. Query the LLM on segment pairs from the rollout
    3. Update the reward model on the new comparisons
    4. Update the policy and value net with PPO
  """
  for round in range(num_rounds):
    print(f"\n{'='*50}")
    print(f"Round {round_num + 1}/{num_rounds}")
    print(f"{'='*50}")

    print("Collecting Rollouts")

    rollout = collect_rollout(env, policy, value_net, reward_model, rollout_steps=rollout_steps)

    # query LLM
    print(f"Querying LLM on {COMPARISONS_PER_ROUND} segment pairs...")
    segments = rollout["segments"]

    new_comparisons = 0
    for i in range(COMPARISONS_PER_ROUND):
      if len(segments) < 2:
        break

      idx_a, idx_b = np.random.choice(len(segments), size=2, replacement=False)
      seg_a, seg_b = segments[idx_a], segments[idx_b]

      preference, explanation, raw = get_llm_preference(seg_a, seg_b)

      if preference in ["A", "B"]:
        buffer.add(seg_a, seg_b, preference)
        new_comparisons += 1
        print(f"Comparison {new_comparisons}: LLM prefers {preference} — {explanation[:60]}...")

  print(f"Added {new_comparisons} new comparisons. Buffer size: {len(buffer)}")


  if len(buffer) >= 5:
    print(f"Training reward model on {len(buffer)} comparisons...")
    rm_loss = train_reward_model(reward_model, buffer, num_epochs=RM_TRAIN_EPOCHS, lr=LR_REWARD)
    print(f"  RM final loss: {rm_losses[-1]:.4f}")
  else:
    print("Not enough comparisons yet to train reward model (need >= 5)")


  print("Running PPO update...")
  advantages, returns = compute_gae(rollout["rewards"], rollout["values"], rollout["dones"])

  ppo_update(
    policy, value_net,
    rollout["observations"],
    rollout["actions"],
    rollout["logprobs"],
    advantages,
    returns
  )

  avg_rm_reward = np.mean(rollout["rewards"])
  if segments:
    avg_upright = np.mean([s["time_upright"] for s in segments])
  else:
    avg_upright = 0

  print(f"\nRound {round_num + 1} summary:")
  print(f"  Avg reward model score: {avg_rm_reward:.4f}")
  print(f"  Avg time upright:       {avg_upright:.1f}/{SEGMENT_LENGTH} steps")
  print(f"  Segments collected:     {len(segments)}")
  print(f"  Buffer size:            {len(buffer)}")

